In [ ]:
### knowledge graph 와 web 을 혼합한 RAG 구현

In [ ]:
### Finance 관련된 knowledge Graph 에서 데이터를 가져오는 RAG 클래스

## 1. query 에서 knowledge graph Mock API 를 호출하기 위한 중간 json 결과를 만드는 prompot
entity_extract_template =
"""
You are given a Query and Query Time. Do the following:

1) Determine the domain the query is about. The domain should be one of the following: "finance", "sports", "music", "movie", "encyclopedia". If none of the domain applies, use "other". Use "domain" as the key in the result json.

2) Extract structured information from the query. Include different keys into the result json depending on the domains, and put them DIRECTLY in the result json. Here are the rules:

For `finance` queries, these are possible keys:
- `market_identifier`: stock identifiers including individual company names, stock symbols.
- `metric`: financial metrics that the query is asking about. This must be one of the following: `price`, `dividend`, `P/E ratio`, `EPS`, `marketCap`, and `other`.
- `datetime`: time frame that query asks about. When datetime is not explicitly mentioned, use `Query Time` as default.


Return the results in a FLAT json.

*NEVER include ANY EXPLANATION or NOTE in the output, ONLY OUTPUT JSON*
"""


## 2. Mock KG Query Engine

class KGQueryEngine:
    def query(self, query):
        generated_query, is_finance = self.generate_query(query)

        if is_finance:
            kg_results = self.get_finance_kg_results(generated_query)
        else:
            kg_results = ""

        return kg_results, is_finance

    def generate_query( self, query ):
        llm_input = prompt_generator( query )
        completion = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=llm_input
        ).choices[0].message.content

        try:
            completion = json.loads(completion)
        except:
            completion = extract_json_objects(completion)

        if "domain" in completion.keys():
            domain = completion["domain"]
            is_finance = domain == "finance"
        else:
            is_finance = False

        return completion, is_finance

    def get_finance_kg_results( self, generated_query ):
        formatted_time_list = []
        if 'datetime' in generated_query:
            datetime_list = generated_query['datetime'].split(' - ')
            for datetime in datetime_list:
                formatted_time_list.append(convert_to_standard_format(datetime.strip()))

        kg_results = []
        res = ""
        if "market_identifier" in generated_query.keys() and generated_query["market_identifier"] is not None:
            if isinstance(generated_query["market_identifier"], str):
                company_names = generated_query["market_identifier"].split(",")
            else:
                company_names = generated_query["market_identifier"]

            for company_name in company_names:
                try:
                    res = api.finance_get_company_name(company_name)["result"]

                    if res == []:
                        ticker_name = company_name.upper()
                    else:
                        ticker_name = api.finance_get_ticker_by_name(res[0])["result"]

                    if generated_query['metric'].lower().strip() == 'price':
                        response = api.finance_get_price_history(ticker_name)['result']
                    elif generated_query['metric'].lower().strip() == 'dividend':
                        response = api.finance_get_dividends_history(ticker_name)['result']
                    elif generated_query['metric'].lower().strip() == 'p/e ratio':
                        response = api.finance_get_pe_ratio(ticker_name)['result']
                    elif generated_query['metric'].lower().strip() == 'eps':
                        response = api.finance_get_eps(ticker_name)["result"]
                    elif generated_query['metric'].lower().strip() == 'marketcap' :
                        response = api.finance_get_market_capitalization(ticker_name)['result']
                    else:
                        response = api.finance_get_info(ticker_name)['result']
                        metric_value = get_metric_from_response(response, generated_query['metric'])
                        if metric_value is not None:
                            response = metric_value

                    try:
                        for formatted_time in formatted_time_list:
                            if formatted_time in response:
                                filtered_response = copy.deepcopy(response[formatted_time])
                            elif add_one_day(formatted_time) in response:
                                filtered_response = copy.deepcopy(response[add_one_day(formatted_time)])
                            elif subtract_one_day(formatted_time) in response:
                                filtered_response = copy.deepcopy(response[subtract_one_day(formatted_time)])
                            else:
                                filtered_response = copy.deepcopy(response)
                            kg_results.append({company_name + " " + generated_query["metric"]: filtered_response, 'time': formatted_time})
                    except:
                        kg_results.append({company_name + " " + generated_query["metric"]: response})

                except Exception as e:
                    print("Fail to parse the generated query")
                    pass

        kg_results = "<DOC>\n".join([str(res) for res in kg_results]) if len(kg_results) > 0 else ""
        return  kg_results

    def prompt_generator(self, query):
        user_message = ""
        user_message += f"Query: {query}\n"

        llm_input = [
          {"role": "system", "content": entity_extract_template},
          {"role": "user", "content": user_message},
        ]

        return llm_input



## 3.3 Reader 클래스 구현 (이전 Web based 의 Reader 와 동일)

from openai import OpenAI
oai_client = OpenAI()

class Reader:
    def __init__(self):
        self.system_prompt = """
            You are provided with a question and various references.
            Your task is to answer the question succinctly, using the fewest words possible.
            If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
            There is no need to explain the reasoning behind your answers.
        """

    def generate_response( self, question: str, top_k_chunks: list ) -> str:      # Generate answer from context.
        llm_input = self.prompt_generator( question, top_k_chunks )
        completion = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=llm_input
      ).choices[0].message.content
      return completion

    def prompt_generator(self, query, top_k_chunks):
        user_message = ""
        references = ""

        if len(top_k_chunks) > 0:
            references += "# References \n"
            # Format the top sentences as references in the model's prompt template.
            for chunk_id, chunk in enumerate(top_k_chunks):
                references += f"- {chunk.strip()}\n"

        references = references[:MAX_CONTEXT_REFERENCES_LENGTH]
        # Limit the length of references to fit the model's input size.

        user_message += f"{references}\n------\n\n"
        user_message += f"Using only the references listed above, answer the following question: \n"
        user_message += f"Question: {query}\n"

        llm_input = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message},
        ]

        return llm_input

In [ ]:
### Mock KG + LLM 을 사용하는 RAG 클래스 구현

class RAGWithKG:
    def __init__(self):
        self.kg_query_engine = KGQueryEngine()
        self.reader = Reader()

    def inference(self, query):
        # 1. retrieve relevant kg results
        kg_results, is_finance = self.kg_query_engine.query( query )

        # 2. answer the question based on the retrieved chunks
        answer = self.reader.generate_response( query, [kg_results] )

        return answer, kg_results

In [ ]:
### Mock KG + Web Search Result + LLM 을 사용하는 RAG 클래스 구현

from llama_index.core.schema import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

def parse_htmls(search_results):
    all_documents = []

    # Process each HTML text from the search results to extract text content.
    for html_text in search_results:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup( html_text["page_result"], features="lxml" )
        text = soup.get_text(" ", strip=True)  # Use space as a separator, strip whitespaces
        all_documents.append(text)

    return all_documents


class LlamaIndexRetriever:
    def __init__(self):
        self.parser = SentenceSplitter( chunk_size=512, chunk_overlap=0 )

    def retrieve( self, query, search_results, topk ):
        documents = []

        for document in parse_htmls( search_results ):
            if not document:
                # If no text is extracted, add an empty string as a placeholder.
                documents.append( Document(text="") )
            else:
                documents.append( Document(text=document) )

        # Split documents into chunks & Create vector index
        base_index = VectorStoreIndex.from_documents( documents = documents, transformations=[self.parser] )

        # Execute query
        base_retriever = base_index.as_retriever( similarity_top_k=topk )
        retrieved_nodes = base_retriever.retrieve( query )
        retrieved_results = [ retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes ]

        return retrieved_results


class RAGWithSRKG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine()
        self.reader = Reader()

    def inference( self, query, search_results, topk ):
        # 1. retrieve relevant chunks
        retrieved_results = self.retriever.retrieve( query, search_results, topk )

        # 2. retrieve relevant kg results
        kg_results, is_finance = self.kg_query_engine.query( query )

        # combined_results = [kg_results]
        # combined_results.extend(retrieved_results)
        if is_finance:
          combined_results = [kg_results]
        else:
          combined_results = retrieved_results

        # 3. answer the question based on the retrieved chunks
        answer = self.reader.generate_response( query, combined_results )

        return answer, combined_results